In [1]:
import os
import numpy as np
import pandas as pd
import utils
import string
ALPHABET_LIST = list(string.ascii_uppercase)

In [2]:
filepath = '/root/cernbox/backups/old_database/2023_07_17/pmt_202307171723.csv'
src_dir, filename = os.path.split(filepath)
data = pd.read_csv(filepath)
data

,serial_number,id_pmt_block,module,module_in_legacy_tilecal,pos_in_legacy_tilecal,pos_in_phase2_tilecal,beta,HV_nominal,QE,pmt_type,run_number,component_status,current_location,remark,created_at,updated_at,created_by,updated_by
0,110003,NaN,NaN,NaN,NaN,NaN,6.482,708.0,20.9,R11187\r,158.0,NaN,NaN,NaN,2021-03-31 11:24:24,2022-11-16 12:41:44,NaN,NaN
1,110005,NaN,NaN,NaN,NaN,NaN,6.581,687.0,20.6,R11187\r,158.0,NaN,NaN,NaN,2021-03-31 11:24:24,2021-03-31 09:24:24,NaN,NaN
2,110032,NaN,NaN,NaN,NaN,NaN,6.385,733.0,21.0,R11187\r,158.0,NaN,NaN,NaN,2021-03-31 11:24:24,2021-03-31 09:24:24,NaN,NaN
3,110041,NaN,NaN,NaN,NaN,NaN,6.688,641.0,20.5,R11187\r,158.0,NaN,NaN,NaN,2021-03-31 11:24:24,2021-03-31 09:24:24,NaN,NaN
4,110050,NaN,NaN,NaN,NaN,NaN,6.417,667.0,22.0,R11187\r,158.0,NaN,NaN,NaN,2021-03-31 11:24:24,2021-03-31 09:24:24,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10710,BB0017,NaN,NaN,EBA15,23.0,NaN,-1.000,-1.0,NaN,R7877,NaN,NaN,NaN,NaN,2021-03-31 11:24:15,2021-03-31 09:24:15,NaN,NaN
10711,BB0166,NaN,NaN,EBA51,14.0,NaN,-1.000,-1.0,NaN,R7877,NaN,NaN,NaN,NaN,2021-03-31 11:24:15,2021-03-31 09:24:15,NaN,NaN
10712,null01,NaN,NaN,NaN,NaN,NaN,6.776,637.0,23.5,R11187\r,164.0,NaN,NaN,NaN,2021-03-31 11:24:25,2021-03-31 09:24:25,NaN,NaN
10713,XX0018,NaN,NaN,NaN,0.0,NaN,-1.000,642.0,NaN,R7877,NaN,NaN,NaN,NaN,2021-03-31 11:24:15,2021-03-31 09:24:15,NaN,NaN


In [3]:
data['new_id'] = pd.NaT
data['new_type'] = 'pmt'
data['new_owner'] = 'CER'
data['new_location'] = pd.NaT
data['new_status'] = pd.NaT
data['new_serial'] = pd.NaT
data['new_beta'] = np.nan
data['new_hv_nominal'] = np.nan
data['new_qe'] = np.nan
data['new_model'] = pd.NaT
data['new_comments'] = ''
data['new_phase1_position'] = pd.NaT
data['new_detector_location'] = pd.NaT

## General Checks

In [4]:
data.shape

(10715, 31)

In [5]:
data.dtypes

serial_number                       object
id_pmt_block                       float64
module                             float64
module_in_legacy_tilecal            object
pos_in_legacy_tilecal              float64
pos_in_phase2_tilecal              float64
beta                               float64
HV_nominal                         float64
QE                                 float64
pmt_type                            object
run_number                         float64
component_status                    object
current_location                    object
remark                              object
created_at                          object
updated_at                          object
created_by                         float64
updated_by                         float64
new_id                      datetime64[ns]
new_type                            object
new_owner                           object
new_location                datetime64[ns]
new_status                  datetime64[ns]
new_serial 

In [6]:
null_count = data.isnull().sum()
#Selects columns that have non null values
non_null_columns = null_count[null_count < len(data)].index
data  = data[non_null_columns]
null_count

serial_number                   0
id_pmt_block                10715
module                      10715
module_in_legacy_tilecal     1044
pos_in_legacy_tilecal         448
pos_in_phase2_tilecal       10715
beta                            0
HV_nominal                      0
QE                          10315
pmt_type                        0
run_number                  10315
component_status            10628
current_location            10389
remark                      10536
created_at                      0
updated_at                      0
created_by                  10715
updated_by                  10715
new_id                      10715
new_type                        0
new_owner                       0
new_location                10715
new_status                  10715
new_serial                  10715
new_beta                    10715
new_hv_nominal              10715
new_qe                      10715
new_model                   10715
new_comments                    0
new_phase1_pos

## Check comments for module information

## Serial number valid?

### Serials have multiple regex matches

We either have serial that match the expected regex, serial that are full numerical that were made numeric with the mapping of the first 2 letters to their index in the alphabet or there is a strange value called null01

In [7]:
correct_regex = '[A-Z]{2}[0-9]{4}'
correct_match = utils.match_regex('serial_number', correct_regex, data)
number_regex = '[0-9]{6}'
number_match = utils.match_regex('serial_number', number_regex, data)
data.loc[~(correct_match | number_match)]

,serial_number,module_in_legacy_tilecal,pos_in_legacy_tilecal,beta,HV_nominal,QE,pmt_type,run_number,component_status,current_location,remark,created_at,updated_at,new_type,new_owner,new_comments
10712,null01,NaN,NaN,6.776,637.0,23.5,R11187\r,164.0,NaN,NaN,NaN,2021-03-31 11:24:25,2021-03-31 09:24:25,pmt,CER,


In [8]:
utils.is_unique('serial_number', data)

True

### Cleaning

In [9]:
data.loc[:, 'new_serial'] = data['serial_number']

/tmp/ipykernel_169411/2761155523.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data.loc[:, 'new_serial'] = data['serial_number']


Had to remove cell bellow due to the logic used to generate the serial numbers. Not necessarily 2 means B and so on.

In [10]:
# def map_back_to_alphabet(serial_number: str):
#     serial_list = list(serial_number)
#     serial_list[0] = ALPHABET_LIST[int(serial_list[0])]
#     serial_list[1] = ALPHABET_LIST[int(serial_list[1])]
#     return ''.join(serial_list)
# data.loc[number_match, 'new_serial'] = data.loc[number_match, 'serial_number'].apply(
#     map_back_to_alphabet
# )

In [11]:
data = data.loc[data['new_serial'] != 'null01']

#### Fixed? (Yes)

In [12]:
utils.is_unique('new_serial', data)

True

## Beta is valid?

In [13]:
beta_is_positive = utils.greater_than('beta', 0, data)
data.loc[~beta_is_positive, 'beta'].value_counts(dropna=False)

beta
-1.000    1013
 0.000     258
-4.406       1
-4.817       1
Name: count, dtype: int64

In [14]:
data.loc[beta_is_positive, 'new_beta'] = data.loc[beta_is_positive, 'beta']

In [15]:
(data['new_beta'] <= 0).any()

False

## HV nominal is valid?

In [16]:
hv_is_positive = utils.greater_than('HV_nominal', 0, data)
data.loc[~hv_is_positive, 'HV_nominal'].value_counts(dropna=False)

HV_nominal
-1.0    13
 0.0     2
Name: count, dtype: int64

In [17]:
data.loc[hv_is_positive, 'new_hv_nominal'] = data.loc[hv_is_positive, 'HV_nominal']

In [18]:
(data['new_hv_nominal'] <= 0).any()

False

## QE is valid?

All non positive QE values are NaN

In [19]:
qe_is_positive = utils.greater_than('QE', 0, data)
data.loc[~qe_is_positive, 'QE'].value_counts(dropna=False)

QE
NaN    10315
Name: count, dtype: int64

In [20]:
data.loc[qe_is_positive, 'new_qe'] = data.loc[qe_is_positive, 'QE']

In [21]:
(data['new_qe'] <= 0).any()

False

## Type is valid?
There are some pmt types that end in `\r` for some reason, does this means something or just multiple typos? I assumed is just a type and replaced it

In [22]:
data['pmt_type'].value_counts()

pmt_type
R7877       10315
R11187\r      395
R11187          4
Name: count, dtype: int64

In [23]:
data.loc[:, 'new_model'] = data['pmt_type'].str.replace('\r', '')
data['new_model'].value_counts()

new_model
R7877     10315
R11187      399
Name: count, dtype: int64

## Build Detector Location and P1 Position

Detector location and P1 Position have 2 cases
- The data is correctly inserted in the expected position
- The data is inserted into the comments.

Looking at the comments, there are some possiblities of comments:
1. \{module\} PMT\{position\} \{e-Log link\}

Example: LBC55 PMT 34 https://pcata007.cern.ch/elog/Maintenance/4264'

2. \{section\} \{section_number\} pmt \{position\}

Example: EBC 59 pmt 13

3. PMT block \{PMT block ID\}\n\{section\} MD\{section_number\} pos \{position\}

Example: PMT block 601744\nEB MD1 pos 5

### Case 1 Remarks

In [24]:
case1_phase1_position_re = r'(PMT[0-9][0-9]|PMT [0-9][0-9]|PMT [0-9]|PMT[0-9])'
case1_detector_location_re = r'(EBA|EBC|LBA|LBC)(6[0-4]|[0-5][0-9])'
select_case1_remarks = data['remark'].str.contains(case1_detector_location_re, na=False)
select_case1_columns =  ["remark", 'module_in_legacy_tilecal', 'pos_in_legacy_tilecal', 'component_status', 'current_location']
print(f'{sum(select_case1_remarks)} remarks with module info')
data.loc[select_case1_remarks, select_case1_columns]

/tmp/ipykernel_169411/161007346.py:3: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  select_case1_remarks = data['remark'].str.contains(case1_detector_location_re, na=False)


42 remarks with module info


,remark,module_in_legacy_tilecal,pos_in_legacy_tilecal,component_status,current_location
19,EBA61 PMT17 https://pcata007.cern.ch/elog/Main...,NaN,NaN,NaN,UX15
64,LBA37 PMT2 https://pcata007.cern.ch/elog/Maint...,NaN,NaN,NaN,UX15
67,EBA53 PMT2 https://pcata007.cern.ch/elog/Maint...,NaN,NaN,NaN,UX15
68,EBA34 PMT44 https://pcata007.cern.ch/elog/Main...,NaN,NaN,NaN,UX15
71,EBA18 PMT34 https://pcata007.cern.ch/elog/Main...,NaN,NaN,NaN,UX15
79,EBA25 PMT05 https://pcata007.cern.ch/elog/Main...,NaN,NaN,NaN,UX15
80,LBA59 PMT37 https://pcata007.cern.ch/elog/Main...,NaN,NaN,NaN,UX15
81,LBA35 PMT4 https://pcata007.cern.ch/elog/Maint...,NaN,NaN,NaN,UX15
112,EBC61 PMT14 https://pcata007.cern.ch/elog/Main...,NaN,NaN,NaN,UX15
135,LBC55 PMT 34 https://pcata007.cern.ch/elog/Mai...,NaN,NaN,NaN,UX15


In [25]:
module_rename_cols = {
    0: 'section',
    1: 'section_number',
    'level_0': 'row'
}
extracted_modules = data['remark'].str.extractall(case1_detector_location_re) \
    .reset_index(level='match').rename(module_rename_cols, axis=1)
extracted_modules['module'] = extracted_modules['section']+extracted_modules['section_number']


pos_rename_cols = {
    0: 'pmt',
    'level_0': 'row'
}
extracted_positions = data['remark'].str.extractall(case1_phase1_position_re) \
    .reset_index(level='match').rename(pos_rename_cols, axis=1)
extracted_positions['pmt'] = extracted_positions['pmt'].str.replace(' ', '')
positions = extracted_positions['pmt'].str.extractall(r'([0-9][0-9]|[0-9])')\
    .reset_index(level='match', drop=True).rename({0: 'pmt_num'}, axis=1)

module_positions = extracted_modules.join(extracted_positions, rsuffix='_module', lsuffix='_pos')\
    .join(positions)
data.loc[module_positions.index, 'remark_module'] = module_positions['module']
data.loc[module_positions.index, 'remark_position'] = module_positions['pmt_num']
data.loc[module_positions.index, 'pmt'] = module_positions['pmt']

data['remark'] = data['remark'].replace(case1_phase1_position_re, '', regex=True) \
    .replace(case1_detector_location_re, '', regex=True) \
    .str.lstrip(' ') \
    .str.rstrip(' ')
data.loc[select_case1_remarks, select_case1_columns + ['remark_module', 'remark_position']]

,remark,module_in_legacy_tilecal,pos_in_legacy_tilecal,component_status,current_location,remark_module,remark_position
19,https://pcata007.cern.ch/elog/Maintenance/3615,NaN,NaN,NaN,UX15,EBA61,17
64,https://pcata007.cern.ch/elog/Maintenance/2101,NaN,NaN,NaN,UX15,LBA37,2
67,https://pcata007.cern.ch/elog/Maintenance/1925,NaN,NaN,NaN,UX15,EBA53,2
68,https://pcata007.cern.ch/elog/Maintenance/3660,NaN,NaN,NaN,UX15,EBA34,44
71,https://pcata007.cern.ch/elog/Maintenance/3501,NaN,NaN,NaN,UX15,EBA18,34
79,https://pcata007.cern.ch/elog/Maintenance/3745,NaN,NaN,NaN,UX15,EBA25,05
80,https://pcata007.cern.ch/elog/Maintenance/3559,NaN,NaN,NaN,UX15,LBA59,37
81,https://pcata007.cern.ch/elog/Maintenance/2141,NaN,NaN,NaN,UX15,LBA35,4
112,https://pcata007.cern.ch/elog/Maintenance/4217,NaN,NaN,NaN,UX15,EBC61,14
135,https://pcata007.cern.ch/elog/Maintenance/4264,NaN,NaN,NaN,UX15,LBC55,34


### Case 2 Remarks

In [26]:
case2_phase1_position_re = r'(pmt[0-9][0-9]|pmt [0-9][0-9]|pmt [0-9]|pmt[0-9])'
case2_detector_location_re = r'(EBA|EBC|LBA|LBC) (6[0-4]|[0-5][0-9])'
select_case2_remarks = data['remark'].str.contains(case2_phase1_position_re, na=False)
print(f'{sum(select_case2_remarks)} remarks with module info')
data.loc[select_case2_remarks, select_case1_columns]

1 remarks with module info


/tmp/ipykernel_169411/3654567762.py:3: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  select_case2_remarks = data['remark'].str.contains(case2_phase1_position_re, na=False)


,remark,module_in_legacy_tilecal,pos_in_legacy_tilecal,component_status,current_location
147,EBC 59 pmt 13,NaN,NaN,NaN,on detector


In [27]:
# Since is just one case
data.loc[147, 'remark'] = pd.NaT
data.loc[147, 'remark_module'] = 'EBC59'
data.loc[147, 'remark_position'] = 13

### Case 3 Remarks
Not sure on how to obtain this data

In [28]:
case3_phase1_position_re = r'PMT block [0-9]{6}\n'
case3_detector_location_re = r'(EBA|EBC|LBA|LBC) (6[0-4]|[0-5][0-9])'
select_case3_remarks = data['remark'].str.contains(case3_phase1_position_re, na=False, regex=True)
print(f'{sum(select_case3_remarks)} remarks with module info')
data.loc[select_case3_remarks, select_case1_columns]

29 remarks with module info


,remark,module_in_legacy_tilecal,pos_in_legacy_tilecal,component_status,current_location
126,PMT block 100398\nEB MD1 pos 7,NaN,NaN,NaN,CERN Testbeam
129,PMT block 301166\nEB MD2 pos 9,NaN,NaN,NaN,CERN Testbeam
132,PMT block 101127\nEB MD3 pos 4,NaN,NaN,NaN,CERN Testbeam
158,PMT block 300661\nEB MD1 pos 4,NaN,NaN,NaN,CERN Testbeam
164,PMT block 601744\nEB MD1 pos 5,NaN,NaN,NaN,CERN Testbeam
165,PMT block 700147\nEB MD3 pos 2,NaN,NaN,NaN,CERN Testbeam
180,PMT block 300601\nEB MD2 pos 1,NaN,NaN,NaN,CERN Testbeam
187,PMT block 700766\nEB MD1 pos 1,NaN,NaN,NaN,CERN Testbeam
193,PMT block 601968\nEB MD2 pos 4,NaN,NaN,NaN,CERN Testbeam
239,PMT block 700203\nEB MD2 pos 8,NaN,NaN,NaN,CERN Testbeam


### Misc cases

In [29]:
def build_p1_variables(row):

    remark_module = row['remark_module']
    old_module = row["module_in_legacy_tilecal"]
    remark_module_empty = utils.is_empty(remark_module)
    old_module_empty = utils.is_empty(old_module)
    if (not remark_module_empty) and (not old_module_empty):
        if remark_module == old_module:
            row['new_phase1_position'] = remark_pos
        else:
            raise ValueError(f'Not equal values remark_pos {remark_pos} and old_pos {old_pos}')
    elif not remark_module_empty:
        row['new_detector_location'] = remark_module
    elif old_module_empty:
        row['new_detector_location'] = pd.NaT
    else:
        row['new_detector_location'] = str(old_module).upper()
    
    remark_pos = row['remark_position']
    old_pos = row["pos_in_legacy_tilecal"]
    remark_pos_empty = utils.is_empty(remark_pos)
    old_pos_empty = utils.is_empty(old_pos)
    if (not remark_pos_empty) and (not old_pos_empty):
        if remark_pos == old_pos:
            row['new_phase1_position'] = remark_pos
        else:
            raise ValueError(f'Not equal values remark_pos {remark_pos} and old_pos {old_pos}')
    elif not remark_pos_empty:
        row['new_phase1_position'] = remark_pos
    elif old_pos_empty:
        row['new_phase1_position'] = pd.NaT
    else:
        row['new_phase1_position'] = int(old_pos)
    return row

In [30]:
empty_module = data['module_in_legacy_tilecal'].apply(utils.is_empty)
empty_pos = data['pos_in_legacy_tilecal'].apply(utils.is_empty)
data = data.apply(build_p1_variables, axis=1)
data

,serial_number,module_in_legacy_tilecal,pos_in_legacy_tilecal,beta,HV_nominal,QE,pmt_type,run_number,component_status,current_location,...,new_serial,new_beta,new_hv_nominal,new_qe,new_model,remark_module,remark_position,pmt,new_detector_location,new_phase1_position
0,110003,NaN,NaN,6.482,708.0,20.9,R11187\r,158.0,NaN,NaN,...,110003,6.482,708.0,20.9,R11187,NaN,NaN,NaN,NaT,NaT
1,110005,NaN,NaN,6.581,687.0,20.6,R11187\r,158.0,NaN,NaN,...,110005,6.581,687.0,20.6,R11187,NaN,NaN,NaN,NaT,NaT
2,110032,NaN,NaN,6.385,733.0,21.0,R11187\r,158.0,NaN,NaN,...,110032,6.385,733.0,21.0,R11187,NaN,NaN,NaN,NaT,NaT
3,110041,NaN,NaN,6.688,641.0,20.5,R11187\r,158.0,NaN,NaN,...,110041,6.688,641.0,20.5,R11187,NaN,NaN,NaN,NaT,NaT
4,110050,NaN,NaN,6.417,667.0,22.0,R11187\r,158.0,NaN,NaN,...,110050,6.417,667.0,22.0,R11187,NaN,NaN,NaN,NaT,NaT
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10709,BB0015,EBA15,14.0,-1.000,-1.0,NaN,R7877,NaN,NaN,NaN,...,BB0015,NaN,NaN,NaN,R7877,NaN,NaN,NaN,EBA15,14
10710,BB0017,EBA15,23.0,-1.000,-1.0,NaN,R7877,NaN,NaN,NaN,...,BB0017,NaN,NaN,NaN,R7877,NaN,NaN,NaN,EBA15,23
10711,BB0166,EBA51,14.0,-1.000,-1.0,NaN,R7877,NaN,NaN,NaN,...,BB0166,NaN,NaN,NaN,R7877,NaN,NaN,NaN,EBA51,14
10713,XX0018,NaN,0.0,-1.000,642.0,NaN,R7877,NaN,NaN,NaN,...,XX0018,NaN,642.0,NaN,R7877,NaN,NaN,NaN,NaT,0


## Location is Valid?
There are multple locations that map to the same place, those values need to be standardized.
There are some unknown values:

In [31]:
data.loc[:, 'new_location'] = data['current_location'].str.replace(' ', '_').str.lower()
data['new_location'].value_counts(dropna=False)

new_location
NaN                    10388
cern_175                 130
ux15                      40
bratislava                29
cern_testbeam             29
pisa_q&a_grid             23
cern_171                  21
pisa                      11
lhcb                      11
bratislava_?              10
should_be_at_cern         10
pisa_long_term_test        4
pisa_hvad_test             2
cern_174                   1
on_detector                1
cluj                       1
cenr_171                   1
bin                        1
cern_171_                  1
Name: count, dtype: int64

In [32]:
location_map = {
    'cern_171': 'B171',
    'cenr_171': 'B171',
    'cern_171_': 'B171',
    'cern_174': 'B175',
    'cern_175': 'B175',
    'pisa': 'Universita & INFN, Pisa',
    'ux15': 'UX15',
    'on_detector': 'ATLAS',
    'cern_testbeam': 'Testbeam',
    'lhcb': 'LHCb',
    'cluj': 'TIM, Cluj Napoca',
    'bratislava': 'Comenius University Bratislava',
    'bin': 'Unknown'
}

def uniformize_location_edge_cases(row: pd.Series) -> str:
    location = row['new_location']
    if utils.is_empty(location):
        if utils.is_empty(row['new_detector_location']) and utils.is_empty(row['new_phase1_position']):
            row['new_location'] = 'Unknown'
        else:
            row['new_location'] = 'UX15'
    elif location.startswith('pisa'):
        splitted = location.split('_')
        row['new_comments'] += '_'.join(splitted[1:])
        row['new_location'] = location_map['pisa']
    elif location == 'should_be_at_cern':
        row['new_location'] = 'Unknown'
        if row['new_comments']:
            row['new_comments'] += ';Should be in CERN'
        else:
            row['new_comments'] += 'Should be in CERN'
    elif location == 'bratislava_?':
        row['new_location'] = 'Unknown'
        if row['new_comments']:
            row['new_comments'] += ';Should be in Bratislava'
        else:
            row['new_comments'] += 'Should be in Bratislava'
    else:
        row['new_location'] = location
    
    return row


def uniformize_locations(row: pd.Series) -> pd.Series:
    try:
        row['new_location'] = location_map[row['new_location']]
        return row
    except KeyError:
        return uniformize_location_edge_cases(row)

data = data.apply(uniformize_locations, axis=1)
data

,serial_number,module_in_legacy_tilecal,pos_in_legacy_tilecal,beta,HV_nominal,QE,pmt_type,run_number,component_status,current_location,...,new_beta,new_hv_nominal,new_qe,new_model,remark_module,remark_position,pmt,new_detector_location,new_phase1_position,new_location
0,110003,NaN,NaN,6.482,708.0,20.9,R11187\r,158.0,NaN,NaN,...,6.482,708.0,20.9,R11187,NaN,NaN,NaN,NaT,NaT,Unknown
1,110005,NaN,NaN,6.581,687.0,20.6,R11187\r,158.0,NaN,NaN,...,6.581,687.0,20.6,R11187,NaN,NaN,NaN,NaT,NaT,Unknown
2,110032,NaN,NaN,6.385,733.0,21.0,R11187\r,158.0,NaN,NaN,...,6.385,733.0,21.0,R11187,NaN,NaN,NaN,NaT,NaT,Unknown
3,110041,NaN,NaN,6.688,641.0,20.5,R11187\r,158.0,NaN,NaN,...,6.688,641.0,20.5,R11187,NaN,NaN,NaN,NaT,NaT,Unknown
4,110050,NaN,NaN,6.417,667.0,22.0,R11187\r,158.0,NaN,NaN,...,6.417,667.0,22.0,R11187,NaN,NaN,NaN,NaT,NaT,Unknown
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10709,BB0015,EBA15,14.0,-1.000,-1.0,NaN,R7877,NaN,NaN,NaN,...,NaN,NaN,NaN,R7877,NaN,NaN,NaN,EBA15,14,UX15
10710,BB0017,EBA15,23.0,-1.000,-1.0,NaN,R7877,NaN,NaN,NaN,...,NaN,NaN,NaN,R7877,NaN,NaN,NaN,EBA15,23,UX15
10711,BB0166,EBA51,14.0,-1.000,-1.0,NaN,R7877,NaN,NaN,NaN,...,NaN,NaN,NaN,R7877,NaN,NaN,NaN,EBA51,14,UX15
10713,XX0018,NaN,0.0,-1.000,642.0,NaN,R7877,NaN,NaN,NaN,...,NaN,642.0,NaN,R7877,NaN,NaN,NaN,NaT,0,UX15


In [33]:
data['new_location'].value_counts(dropna=False)

new_location
UX15                              10199
Unknown                             250
B175                                131
Universita & INFN, Pisa              40
Comenius University Bratislava       29
Testbeam                             29
B171                                 23
LHCb                                 11
ATLAS                                 1
TIM, Cluj Napoca                      1
Name: count, dtype: int64

## Status is valid?

In [34]:
data['new_status'] = data['component_status'].str.lower().str.replace(' ', '_')
data['new_status'].value_counts(dropna=False)

new_status
NaN                  10627
removed                 57
being_tested            20
broken                   4
high_dark_current        2
bad_linearity            2
hv_problems              1
removed,_broken          1
Name: count, dtype: int64

In [35]:
status_map = {
    'removed': 'Stored',
    'being_tested': 'Testing',
    'broken': 'Broken',
    'removed,_broken': 'Broken'
}

def uniformize_status_edge_cases(row: pd.Series) -> str:
    status = row['new_status']
    problem_type = ['high_dark_current', 'bad_linearity', 'hv_problems']
    if utils.is_empty(status):
        row['new_status'] = 'Unknown'
    elif status in problem_type:
        row['new_status'] = 'Failure'
        row['new_comments'] += ';' + status
    else:
        row['new_status'] = status
    
    return row


def uniformize_status(row: pd.Series) -> pd.Series:
    try:
        row['new_status'] = status_map[row['new_status']]
        return row
    except KeyError:
        return uniformize_status_edge_cases(row)

data = data.apply(uniformize_status, axis=1)
data

,serial_number,module_in_legacy_tilecal,pos_in_legacy_tilecal,beta,HV_nominal,QE,pmt_type,run_number,component_status,current_location,...,new_hv_nominal,new_qe,new_model,remark_module,remark_position,pmt,new_detector_location,new_phase1_position,new_location,new_status
0,110003,NaN,NaN,6.482,708.0,20.9,R11187\r,158.0,NaN,NaN,...,708.0,20.9,R11187,NaN,NaN,NaN,NaT,NaT,Unknown,Unknown
1,110005,NaN,NaN,6.581,687.0,20.6,R11187\r,158.0,NaN,NaN,...,687.0,20.6,R11187,NaN,NaN,NaN,NaT,NaT,Unknown,Unknown
2,110032,NaN,NaN,6.385,733.0,21.0,R11187\r,158.0,NaN,NaN,...,733.0,21.0,R11187,NaN,NaN,NaN,NaT,NaT,Unknown,Unknown
3,110041,NaN,NaN,6.688,641.0,20.5,R11187\r,158.0,NaN,NaN,...,641.0,20.5,R11187,NaN,NaN,NaN,NaT,NaT,Unknown,Unknown
4,110050,NaN,NaN,6.417,667.0,22.0,R11187\r,158.0,NaN,NaN,...,667.0,22.0,R11187,NaN,NaN,NaN,NaT,NaT,Unknown,Unknown
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10709,BB0015,EBA15,14.0,-1.000,-1.0,NaN,R7877,NaN,NaN,NaN,...,NaN,NaN,R7877,NaN,NaN,NaN,EBA15,14,UX15,Unknown
10710,BB0017,EBA15,23.0,-1.000,-1.0,NaN,R7877,NaN,NaN,NaN,...,NaN,NaN,R7877,NaN,NaN,NaN,EBA15,23,UX15,Unknown
10711,BB0166,EBA51,14.0,-1.000,-1.0,NaN,R7877,NaN,NaN,NaN,...,NaN,NaN,R7877,NaN,NaN,NaN,EBA51,14,UX15,Unknown
10713,XX0018,NaN,0.0,-1.000,642.0,NaN,R7877,NaN,NaN,NaN,...,642.0,NaN,R7877,NaN,NaN,NaN,NaT,0,UX15,Unknown


In [36]:
data['new_status'].value_counts()

new_status
Unknown    10627
Stored        57
Testing       20
Broken         5
Failure        5
Name: count, dtype: int64

## What are the remarks?

In [37]:
data.loc[~data['remark'].isnull(), 'remark'].value_counts()

remark
Has been in Pisa                                  27
On loan to Yuri Guz                                9
https://pcata007.cern.ch/elog/Maintenance/4217     8
https://pcata007.cern.ch/elog/Maintenance/4284     6
https://pcata007.cern.ch/elog/Maintenance/4099     5
                                                  ..
PMT block 700083\nEB MD5 pos 1                     1
PMT block 101107\nEB MD3 pos 10                    1
PMT block 700166\nEB MD6 pos 2                     1
PMT block 300660\nEB MD6 pos 1                     1
November 2019                                      1
Name: count, Length: 80, dtype: int64

In [38]:
def append_remarks_to_comments(row: pd.Series):
    remark = row['remark']
    comment = row['new_comments']
    if utils.is_empty(remark):
        return row
    elif len(comment) == 0:
        row['new_comments'] = remark
    else:
        row['new_comments'] += ';' + remark

    return row

In [39]:
data = data.apply(append_remarks_to_comments, axis=1)
data['new_comments'].value_counts()

new_comments
                                            10508
q&a_grid                                       17
Should be in Bratislava;Has been in Pisa       10
Has been in Pisa                                9
On loan to Yuri Guz                             9
                                            ...  
PMT block 700166\nEB MD6 pos 2                  1
PMT block 300660\nEB MD6 pos 1                  1
PMT block 700225\nEB MD5 pos 2                  1
PMT block 101056\nEB MD1 pos 11                 1
November 2019                                   1
Name: count, Length: 90, dtype: int64

## Cleaned Data

In [40]:
cleaned_data = data.loc[:, data.columns[data.columns.str.startswith('new')]]
new_column_names = {
    old: '_'.join(old.split('_')[1:])
    for old in cleaned_data.columns
}
cleaned_data.rename(new_column_names, axis=1, inplace=True)
cleaned_data.loc[:, 'id'] = '0' + cleaned_data['serial']
cleaned_data

,type,owner,comments,serial,beta,hv_nominal,qe,model,detector_location,phase1_position,location,status,id
0,pmt,CER,,110003,6.482,708.0,20.9,R11187,NaN,NaN,Unknown,Unknown,0110003
1,pmt,CER,,110005,6.581,687.0,20.6,R11187,NaN,NaN,Unknown,Unknown,0110005
2,pmt,CER,,110032,6.385,733.0,21.0,R11187,NaN,NaN,Unknown,Unknown,0110032
3,pmt,CER,,110041,6.688,641.0,20.5,R11187,NaN,NaN,Unknown,Unknown,0110041
4,pmt,CER,,110050,6.417,667.0,22.0,R11187,NaN,NaN,Unknown,Unknown,0110050
...,...,...,...,...,...,...,...,...,...,...,...,...,...
10709,pmt,CER,,BB0015,NaN,NaN,NaN,R7877,EBA15,14,UX15,Unknown,0BB0015
10710,pmt,CER,,BB0017,NaN,NaN,NaN,R7877,EBA15,23,UX15,Unknown,0BB0017
10711,pmt,CER,,BB0166,NaN,NaN,NaN,R7877,EBA51,14,UX15,Unknown,0BB0166
10713,pmt,CER,,XX0018,NaN,642.0,NaN,R7877,NaT,0,UX15,Unknown,0XX0018


In [41]:
empty_comment = cleaned_data['comments'].str.len() == 0
cleaned_data.loc[empty_comment,'comments'] = pd.NaT
cleaned_data

,type,owner,comments,serial,beta,hv_nominal,qe,model,detector_location,phase1_position,location,status,id
0,pmt,CER,NaN,110003,6.482,708.0,20.9,R11187,NaN,NaN,Unknown,Unknown,0110003
1,pmt,CER,NaN,110005,6.581,687.0,20.6,R11187,NaN,NaN,Unknown,Unknown,0110005
2,pmt,CER,NaN,110032,6.385,733.0,21.0,R11187,NaN,NaN,Unknown,Unknown,0110032
3,pmt,CER,NaN,110041,6.688,641.0,20.5,R11187,NaN,NaN,Unknown,Unknown,0110041
4,pmt,CER,NaN,110050,6.417,667.0,22.0,R11187,NaN,NaN,Unknown,Unknown,0110050
...,...,...,...,...,...,...,...,...,...,...,...,...,...
10709,pmt,CER,NaN,BB0015,NaN,NaN,NaN,R7877,EBA15,14,UX15,Unknown,0BB0015
10710,pmt,CER,NaN,BB0017,NaN,NaN,NaN,R7877,EBA15,23,UX15,Unknown,0BB0017
10711,pmt,CER,NaN,BB0166,NaN,NaN,NaN,R7877,EBA51,14,UX15,Unknown,0BB0166
10713,pmt,CER,NaN,XX0018,NaN,642.0,NaN,R7877,NaT,0,UX15,Unknown,0XX0018


In [42]:
cleaned_data.isnull().sum()

type                     0
owner                    0
comments             10508
serial                   0
beta                  1273
hv_nominal              15
qe                   10315
model                    0
detector_location     1000
phase1_position        404
location                 0
status                   0
id                       0
dtype: int64

In [43]:
cleaned_data['location'].value_counts(dropna=False)

location
UX15                              10199
Unknown                             250
B175                                131
Universita & INFN, Pisa              40
Comenius University Bratislava       29
Testbeam                             29
B171                                 23
LHCb                                 11
ATLAS                                 1
TIM, Cluj Napoca                      1
Name: count, dtype: int64

## Saving data

In [44]:
now = pd.Timestamp.now()
cleaned_filename = f'{now.strftime("%Y_%m_%d_%H_%M_%S")}_cleaned_{filename}'
src_dir, _ = os.path.split(filepath)
cleaned_filepath = os.path.join(
    src_dir,
    'cleaned',
    cleaned_filename
)
print(cleaned_filepath)
cleaned_data.to_csv(cleaned_filepath, index=False)

/root/cernbox/backups/old_database/2023_07_17/cleaned/2023_11_16_10_38_31_cleaned_pmt_202307171723.csv
